# 검수 마스크를 활용한 용종 분할 재학습

같은 검수 사진과 기준 모델에서 출발해 처음 후보 마스크와 사람이 확인한 마스크의 학습 효과를 짝지어 비교합니다.

**GPU 선택 → 모두 실행 → 요청 시 검수 ZIP과 기준 모델 선택 → 결과 ZIP**

첫 입력은 앱에서 내보낸 `review_results.zip`입니다. 두 번째는 앱에서 실제 사용한 `baseline_best.pt`입니다. 마스크에 실제 변경이 있는 경우에만 가중치를 요청할 수 있습니다.

공개 Kvasir-SEG 연구·교육 자료를 사용합니다. [공식 배포 및 이용 조건](https://datasets.simula.no/kvasir-seg/) · [Jha et al., 2020](https://doi.org/10.1007/978-3-030-37734-2_37)


## 01 · 실행 설정

Colab에서 GPU를 고르고 **모두 실행**을 누릅니다. 같은 런타임의 파일을 자동으로 찾고, 없으면 선택 창이 열립니다. `PROFILE=smoke`로 CPU와 합성 자료에서 점검할 수 있습니다.

In [ ]:
# @title 후속 실험 설정
PROFILE = "research"  # @param ["research", "smoke"]
SELECT_REVIEW_ZIP = False  # @param {type:"boolean"}
REVIEW_ZIP_PATH = ""  # @param {type:"string"}
CHECKPOINT_PATH = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
LOCAL_ARCHIVE = ""  # @param {type:"string"}
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}

from pathlib import Path, PurePosixPath
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile
from collections import Counter, defaultdict

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if PROFILE not in {"research", "smoke"}:
    raise ValueError("PROFILE은 research 또는 smoke를 선택하세요.")
SMOKE = PROFILE == "smoke"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
packages = ["numpy", "pandas", "Pillow", "scipy", "matplotlib"]
missing = []
for package in packages:
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from IPython.display import Markdown, display

try:
    import torch
    import torchvision
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision.models.segmentation import LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large
except (ImportError, OSError, RuntimeError) as error:
    raise RuntimeError("Colab 기본 GPU 런타임의 torch와 torchvision 조합을 확인하세요.") from error

DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
if MOUNT_DRIVE:
    if not IN_COLAB:
        raise ValueError("MOUNT_DRIVE는 Colab에서만 사용할 수 있습니다.")
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(
    "/content/drive/MyDrive/kvasir_finetune" if MOUNT_DRIVE else
    "/content/kvasir_finetune" if IN_COLAB else "kvasir_finetune_work"
).resolve()
CACHE = Path("/content/polyp_audit_cache") if IN_COLAB else ROOT / "cache"
INPUTS = ROOT / "inputs"
CACHE.mkdir(parents=True, exist_ok=True)
INPUTS.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "protocol": "reviewed-label-finetune-1.0", "seed": 2026, "side": 64 if SMOKE else 256,
    "split_fractions": {"train": 0.70, "val_select": 0.10, "val_audit": 0.10, "test": 0.10},
    "group_dhash_distance": 6, "group_rgb_mae": 0.08,
    "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
    "probability_threshold": 0.50, "boundary_tolerance_px": 2, "failure_dice": 0.60,
    "epochs": 2 if SMOKE else 5, "batch_size": 4, "workers": 0,
    "backbone_lr": 5e-5, "classifier_lr": 1e-4, "weight_decay": 1e-4,
    "bootstrap_repeats": 2000, "small_delta": 0.01,
}
ENV = {"device": DEVICE.type, "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else None,
       "python": platform.python_version(),
       "versions": {name: metadata.version(name) for name in ["torch", "torchvision", *packages]}}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(path)

def save_csv(frame, filename):
    target = RUN / filename
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)

def save_figure(fig, filename):
    fig.savefig(RUN / "figures" / filename, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)

## 02 · 검수 파일 읽기

앱에서 받은 `review_results.zip`의 검수 기록과 설정을 확인합니다. 다른 ZIP을 직접 고를 때는 첫 셀의 **SELECT_REVIEW_ZIP**에 체크하세요.

In [ ]:
SHA_RE = re.compile(r"[0-9a-f]{64}")
CASE_RE = re.compile(r"[A-Za-z0-9_-]+")
REVIEW_COLUMNS = {"case_id", "status", "actor", "before_mask", "after_mask", "changed_pixels"}
DECISIONS = ("approved", "corrected", "deferred")
SHARED_ROOTS = [Path("/content"), Path("/content/drive/MyDrive")] if IN_COLAB else []

def recent_files(paths):
    present = {path.resolve() for path in paths if path.is_file()}
    return sorted(present, key=lambda path: (-path.stat().st_mtime_ns, str(path)))

def review_candidates():
    return recent_files([
        *INPUTS.glob("*/review_results*.zip"),
        *(path for root in SHARED_ROOTS for path in root.glob("review_results*.zip")),
        *(path for root in SHARED_ROOTS for path in root.glob("kvasir_review/sessions/*/review_results.zip")),
    ])

def checkpoint_candidates():
    patterns = ("baseline_best*.pt", "polyp_audit/runs/*/checkpoints/baseline_best.pt",
                "kvasir_review/training/*/checkpoints/baseline_best.pt", "kvasir_review/uploaded/*/*.pt")
    return recent_files([
        *INPUTS.glob("*/*.pt"),
        *(path for root in SHARED_ROOTS for pattern in patterns for path in root.glob(pattern)),
    ])

def pick_upload(message, suffixes):
    if not IN_COLAB:
        raise FileNotFoundError(message + " 로컬에서는 첫 셀의 경로를 지정하세요.")
    from google.colab import files
    print(message)
    saved = []
    for filename, data in files.upload().items():
        name = Path(filename).name
        if Path(name).suffix.lower() not in suffixes:
            continue
        destination = INPUTS / hashlib.sha256(data).hexdigest()[:16] / name
        destination.parent.mkdir(exist_ok=True)
        destination.write_bytes(data)
        saved.append(destination)
    return saved

def select_review():
    if REVIEW_ZIP_PATH:
        return Path(REVIEW_ZIP_PATH).expanduser()
    paths = [] if SELECT_REVIEW_ZIP else review_candidates()
    if not paths:
        paths = pick_upload("검수 앱에서 내려받은 review_results.zip을 선택하세요. 기준 모델 .pt 파일도 함께 선택할 수 있습니다.",
                            {".zip", ".pt"})
        paths = [path for path in paths if path.suffix.lower() == ".zip"]
        if len(paths) != 1:
            raise ValueError("검수 결과 ZIP 한 개를 선택하세요.")
    if len(paths) > 1:
        print(f"검수 ZIP {len(paths)}개를 발견했습니다. 가장 최근 파일을 사용합니다. 다른 파일은 SELECT_REVIEW_ZIP에서 선택할 수 있습니다.")
    return paths[0]

def check_archive(archive):
    entries = archive.infolist()
    if len(entries) > 3000 or sum(entry.file_size for entry in entries) > 256 * 2**20:
        raise ValueError("검수 결과 ZIP의 파일 수 또는 압축 해제 크기가 예상 범위를 넘습니다.")
    seen = set()
    for entry in entries:
        name = entry.filename
        parts = PurePosixPath(name)
        if parts.is_absolute() or ".." in parts.parts or "\\" in name or name in seen:
            raise ValueError(f"검수 ZIP에 안전하지 않거나 중복된 경로가 있습니다: {name}")
        seen.add(name)
    for name, limit in [("review_log.csv", 4 * 2**20), ("review_setup.json", 2**20)]:
        if name not in seen or archive.getinfo(name).file_size > limit:
            raise ValueError(f"검수 앱에서 받은 review_results.zip이 필요합니다. {name}을 확인하세요.")

def read_review(path):
    if not path.is_file():
        raise FileNotFoundError(f"검수 ZIP을 찾을 수 없습니다: {path}")
    if path.stat().st_size > 256 * 2**20:
        raise ValueError("검수 결과 ZIP의 파일 크기가 예상 범위를 넘습니다.")
    archive_sha = sha256_file(path)
    try:
        with zipfile.ZipFile(path) as archive:
            check_archive(archive)
            setup = json.loads(archive.read("review_setup.json").decode("utf-8"))
            log = pd.read_csv(io.BytesIO(archive.read("review_log.csv")), dtype=str, keep_default_na=False)
    except (zipfile.BadZipFile, json.JSONDecodeError, UnicodeError, pd.errors.ParserError) as error:
        raise ValueError("검수 앱에서 내보낸 review_results.zip 파일을 확인해 주세요.") from error
    if not REVIEW_COLUMNS.issubset(log.columns):
        raise ValueError(f"검수 기록에 필요한 열이 없습니다: {sorted(REVIEW_COLUMNS - set(log.columns))}")
    try:
        cfg, model = setup["config"], setup["model"]
        expected_dataset = setup["dataset"]["archive_sha256"]
        valid_profile = cfg["profile"] == PROFILE
        valid_split_hash = SHA_RE.fullmatch(setup["split_sha256"])
        valid_weight_hash = SHA_RE.fullmatch(model["sha256"])
        valid_dataset_hash = ((expected_dataset is None) if SMOKE else bool(SHA_RE.fullmatch(expected_dataset)))
    except (KeyError, TypeError) as error:
        raise ValueError("review_setup.json에 입력 검증에 필요한 항목이 없습니다.") from error
    if not valid_profile:
        raise ValueError(f"검수 ZIP의 프로필은 {cfg['profile']}입니다. 해당 PROFILE을 선택하세요.")
    if not all((valid_split_hash, valid_weight_hash, valid_dataset_hash)):
        raise ValueError("검수 ZIP의 데이터셋·분할·가중치 SHA-256 형식이 올바르지 않습니다.")
    for field in ("seed", "split_fractions", "group_dhash_distance", "group_rgb_mae", "mean", "std",
                  "probability_threshold"):
        if cfg.get(field) != CONFIG[field]:
            raise ValueError(f"기존 검수 설정의 {field} 값이 예상 정책과 다릅니다.")
    # JSON dictionary order is irrelevant here. Keep the original group allocation ordering.
    side = cfg.get("side")
    if type(side) is not int or not 64 <= side <= 512 or side % 32:
        raise ValueError("검수 입력 해상도는 64~512 사이의 32 배수여야 합니다.")
    CONFIG["side"] = side
    return setup, log, archive_sha

REVIEW_PATH = select_review() if not SMOKE or REVIEW_ZIP_PATH else None
REVIEW_SETUP, REVIEW_LOG, REVIEW_SHA = read_review(REVIEW_PATH) if REVIEW_PATH else (None, None, None)
if REVIEW_PATH:
    print(f"검수 ZIP · {REVIEW_PATH.name} · 기록 {len(REVIEW_LOG)}건")

## 03 · 공식 데이터와 원래 분할 확인

검수 앱이 사용한 데이터 ZIP의 해시와 분할 해시를 대조합니다. 기존 테스트셋과 검수 분할은 이미지 ID와 중복 그룹이 겹치지 않아야 합니다.

In [ ]:
OFFICIAL_URLS = [
    "https://datasets.simula.no/downloads/kvasir-seg.zip",
    "https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip",
    "https://www.kaggle.com/api/v1/datasets/download/debeshjha1/kvasirseg?datasetVersionNumber=3",
]

OFFICIAL_PAGE = "https://datasets.simula.no/kvasir-seg/"

PAPER = "https://doi.org/10.1007/978-3-030-37734-2_37"

EXPECTED_ARCHIVE_SHA256 = "03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7"

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

def pair_files(root, expected):
    def index(folder):
        result = {}
        for path in sorted(folder.iterdir()):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                if path.stem in result:
                    raise ValueError(f"중복 샘플 ID: {path.stem}")
                result[path.stem] = path
        return result

    images, masks = (index(root / directory) for directory in ("images", "masks"))
    if set(images) != set(masks):
        raise ValueError(f"이미지/마스크 ID 불일치: {len(set(images) ^ set(masks))}개")
    if len(images) != expected:
        raise ValueError(f"예상 {expected}쌍, 실제 {len(images)}쌍")
    return [(key, images[key], masks[key]) for key in sorted(images)]

def zip_root(archive, destination):
    with zipfile.ZipFile(archive) as package:
        members = package.infolist()
        if len(members) > 5000 or sum(item.file_size for item in members) > 500_000_000:
            raise ValueError("Kvasir-SEG ZIP의 예상 크기를 초과합니다.")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        package.extractall(destination)
    roots = [folder for folder in destination.rglob("images")
             if folder.is_dir() and (folder.parent / "masks").is_dir()
             and "__MACOSX" not in folder.parts]
    if len(roots) != 1:
        raise ValueError("images/와 masks/가 있는 데이터 폴더를 하나 찾을 수 있어야 합니다.")
    return roots[0].parent

def get_official_data():
    default_archive = CACHE / "kvasir-seg.zip"
    uploaded = Path(LOCAL_ARCHIVE).expanduser() if LOCAL_ARCHIVE else None
    if uploaded is not None and not uploaded.is_file():
        raise FileNotFoundError(f"LOCAL_ARCHIVE의 공식 ZIP을 찾을 수 없습니다: {uploaded}")
    sources = [(uploaded, "user_supplied_official_archive")] if uploaded else []
    if not uploaded:
        if default_archive.exists():
            sources.append((default_archive, "cached_official_archive"))
        ordered_urls = sorted(OFFICIAL_URLS, key=lambda url: url != REVIEW_DATA_SOURCE)
        sources.extend((default_archive, url) for url in ordered_urls)
    cached_provenance = CACHE / "kvasir-seg.source.json"
    errors = []
    for archive, source in sources:
        attempts = 2 if source.startswith("https://") else 1
        for attempt in range(attempts):
            partial = CACHE / "kvasir-seg.zip.part"
            staging = CACHE / "extracting"
            try:
                if source.startswith("https://"):
                    request = urllib.request.Request(source, headers={"User-Agent": "KvasirSEG-Research/1.0"})
                    with urllib.request.urlopen(request, timeout=35) as response, partial.open("wb") as stream:
                        shutil.copyfileobj(response, stream, 1024 * 1024)
                    file_to_verify = partial
                else:
                    file_to_verify = archive
                fingerprint = sha256_file(file_to_verify)
                if EXPECTED_REVIEW_ARCHIVE and fingerprint != EXPECTED_REVIEW_ARCHIVE:
                    raise ValueError(f'검수 앱이 사용한 데이터 ZIP의 SHA-256과 다릅니다: {fingerprint}')
                if source == OFFICIAL_URLS[0] and fingerprint != EXPECTED_ARCHIVE_SHA256:
                    raise ValueError(f"공개 기록과 기본 ZIP의 SHA-256 불일치: {fingerprint}")
                if fingerprint != EXPECTED_ARCHIVE_SHA256:
                    print(f"기본 ZIP과 다른 SHA-256 ({source}): {fingerprint}. 구조 검사를 계속합니다.")
                extracted = CACHE / f"kvasir_{fingerprint}"
                if not extracted.exists():
                    shutil.rmtree(staging, ignore_errors=True)
                    staging.mkdir()
                    origin = zip_root(file_to_verify, staging)
                    pair_files(origin, expected=1000)
                    origin.replace(extracted)
                    shutil.rmtree(staging, ignore_errors=True)
                pairs = pair_files(extracted, expected=1000)
                provenance = {"source": source, "archive_sha256": fingerprint,
                              "matches_simula_primary_sha256": fingerprint == EXPECTED_ARCHIVE_SHA256}
                if file_to_verify == partial:
                    partial.replace(default_archive)
                    write_json(cached_provenance, provenance)
                elif source == "cached_official_archive" and cached_provenance.is_file():
                    recorded = json.loads(cached_provenance.read_text(encoding="utf-8"))
                    if recorded.get("archive_sha256") == fingerprint:
                        provenance["source"] = recorded["source"]
                return pairs, provenance
            except (OSError, ValueError, RuntimeError, zipfile.BadZipFile) as error:
                errors.append(f"{source} ({attempt + 1}/{attempts}): {error}")
                partial.unlink(missing_ok=True)
                shutil.rmtree(staging, ignore_errors=True)
                if attempt + 1 < attempts:
                    time.sleep(2)
    raise RuntimeError("공식 Kvasir-SEG ZIP 준비 실패. Simula 또는 저자의 Kaggle 배포본 ZIP을 내려받아 Colab에 "
                       "업로드한 뒤 LOCAL_ARCHIVE를 지정하고 다시 실행하세요.\n" + "\n".join(errors))

def smoke_fixtures():
    root = CACHE / "smoke_fixture_v1"
    for name in ("images", "masks"):
        (root / name).mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(CONFIG["seed"])
    y, x = np.mgrid[:72, :88]
    for index in range(48):
        height = 72 if index % 2 else 60
        cx, cy = rng.integers([23, 20], [64, 41])
        rx, ry = rng.integers([4, 4], [23, 18])
        mask = ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 < 1
        if index % 3 == 0:
            mask |= ((x - cx - 0.7 * rx) ** 2 + (y - cy) ** 2 < (ry * 0.65) ** 2)
        rgb = rng.integers(20, 180, (72, 88, 3), dtype=np.uint8)
        rgb[mask, 0] = 225
        key = f"fixture_{index:03d}.png"
        Image.fromarray(rgb[:height]).save(root / "images" / key)
        Image.fromarray(mask[:height].astype("uint8") * 255).save(root / "masks" / key)
    return pair_files(root, 48), {"source": "synthetic_smoke_fixtures", "archive_sha256": None}

def image_fingerprint(path):
    with Image.open(path) as image:
        image = image.convert("RGB")
        digest = hashlib.sha256(f"{image.size}".encode() + image.tobytes()).hexdigest()
        thumb = np.asarray(image.resize((32, 32), Image.Resampling.BILINEAR), dtype=np.int16)
        gray = np.asarray(image.convert("L").resize((9, 9), Image.Resampling.BILINEAR))
    horizontal = gray[:8, 1:] > gray[:8, :-1]
    vertical = gray[1:, :8] > gray[:-1, :8]
    dhash = int.from_bytes(np.packbits(np.r_[horizontal.ravel(), vertical.ravel()]).tobytes(), "big")
    return digest, dhash, thumb

def grouped_split(pairs):
    fingerprints = [image_fingerprint(image) for _, image, _ in pairs]
    parents = list(range(len(pairs)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    edges = []
    for i, (digest, dhash, thumb) in enumerate(fingerprints):
        for j in range(i):
            other_digest, other_hash, other_thumb = fingerprints[j]
            distance = (dhash ^ other_hash).bit_count()
            if digest != other_digest and distance > CONFIG["group_dhash_distance"]:
                continue
            mae = float(np.abs(thumb - other_thumb).mean() / 255)
            if digest == other_digest or mae <= CONFIG["group_rgb_mae"]:
                a, b = root(i), root(j)
                parents[max(a, b)] = min(a, b)
                edges.append((pairs[j][0], pairs[i][0], distance, mae, digest == other_digest))
    groups = defaultdict(list)
    for index, (key, _, _) in enumerate(pairs):
        groups[pairs[root(index)][0]].append(key)

    fractions = CONFIG["split_fractions"]
    names = list(fractions)
    targets = np.array(list(fractions.values())) * len(pairs)
    counts = np.zeros(len(names), dtype=int)
    rng = np.random.default_rng(CONFIG["seed"])
    group_ids = rng.permutation(sorted(groups)).tolist()
    group_ids.sort(key=lambda group: -len(groups[group]))  # Stable order after seeded permutation.
    rows = []
    for group in group_ids:
        slot = int(np.argmax(targets - counts))
        rows.extend((key, group, names[slot]) for key in groups[group])
        counts[slot] += len(groups[group])

    split = pd.DataFrame(rows, columns=["id", "group_id", "split"]).sort_values("id")
    split["decoded_rgb_sha256"] = [item[0] for item in fingerprints]
    split = split.reset_index(drop=True)
    assert split.id.is_unique and len(split) == len(pairs)
    assert set(split.split) == set(names) and split.groupby("group_id").split.nunique().max() == 1
    assert split.groupby("decoded_rgb_sha256").split.nunique().max() == 1
    edge_table = pd.DataFrame(edges, columns=["id_a", "id_b", "dhash_distance", "rgb_mae", "exact"])
    return split, edge_table

EXPECTED_REVIEW_ARCHIVE = REVIEW_SETUP["dataset"]["archive_sha256"] if REVIEW_SETUP else None
REVIEW_DATA_SOURCE = REVIEW_SETUP["dataset"].get("source") if REVIEW_SETUP else None
PAIRS, DATA_INFO = smoke_fixtures() if SMOKE else get_official_data()
SPLIT, _ = grouped_split(PAIRS)
SPLIT_SHA = hashlib.sha256(SPLIT.to_csv(index=False).encode()).hexdigest()
if REVIEW_SETUP is not None and SPLIT_SHA != REVIEW_SETUP["split_sha256"]:
    raise ValueError("검수 ZIP의 분할과 다시 만든 분할의 해시가 다릅니다. 검수 앱에서 쓴 데이터 ZIP과 동일한 Colab 설정에서 실행하세요.")
PAIR_BY_ID = {key: (key, image, mask) for key, image, mask in PAIRS}
SPLIT_IDS = {name: SPLIT.loc[SPLIT.split.eq(name), "id"].tolist() for name in CONFIG["split_fractions"]}
SPLIT_BY_ID = SPLIT.set_index("id")[["group_id", "split"]].to_dict("index")
MEAN = np.asarray(CONFIG["mean"], dtype=np.float32)
STD = np.asarray(CONFIG["std"], dtype=np.float32)
print(f"데이터 {len(PAIRS)}쌍 · 검수 후보 분할 {len(SPLIT_IDS['val_audit'])}장 · 평가 분할 {len(SPLIT_IDS['test'])}장")

## 04 · 학습 쌍 확정

사람이 승인하거나 수정한 사례를 두 학습 조건에 넣습니다. 보류 사례는 집계합니다. 픽셀 변경이 없거나 모델 입력 크기에서 변경이 모두 사라지면 학습을 생략합니다.

In [ ]:
def letterbox(image, mask, side):
    width, height = image.size
    ratio = min(side / width, side / height)
    w, h = max(1, round(width * ratio)), max(1, round(height * ratio))
    x, y = (side - w) // 2, (side - h) // 2
    rgb = np.full((side, side, 3), [124, 116, 104], dtype=np.uint8)
    target = np.zeros((side, side), dtype=bool)
    valid = np.zeros_like(target)
    roi = np.s_[y:y + h, x:x + w]
    rgb[roi] = np.asarray(image.resize((w, h), Image.Resampling.BILINEAR))
    target[roi] = np.asarray(mask.resize((w, h), Image.Resampling.NEAREST)) >= 128
    valid[roi] = True
    return rgb, target, valid

def load_pair(key, image_path, mask_path):
    try:
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            if image.size != mask.size:
                raise ValueError(f"이미지 크기 {image.size}, 마스크 크기 {mask.size}")
            return letterbox(image.convert("RGB"), mask.convert("L"), CONFIG["side"])
    except (OSError, ValueError) as error:
        raise RuntimeError(f"{key} 전처리 실패: {error}") from error

class PolypDataset(Dataset):
    def __init__(self, keys, augment=False, epoch=0):
        self.pairs = [PAIR_BY_ID[key] for key in keys]
        self.augment = augment
        self.epoch = epoch

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        key, image_path, mask_path = self.pairs[index]
        rgb, target, valid = load_pair(key, image_path, mask_path)
        if self.augment:
            rng = np.random.default_rng([CONFIG["seed"], self.epoch, index])
            for axis in (0, 1):
                if rng.random() < 0.5:
                    rgb, target, valid = (np.flip(item, axis) for item in (rgb, target, valid))
            adjusted = np.clip(rgb.astype(np.float32) * rng.uniform(0.90, 1.10), 0, 255)
            rgb = np.where(valid[..., None], adjusted, rgb)
        normalized = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        return (torch.from_numpy(normalized.transpose(2, 0, 1).copy()),
                torch.from_numpy(target[None].copy()), torch.from_numpy(valid[None].copy()), key)

def loader(split, epoch=0):
    training = split == "train"
    generator = torch.Generator().manual_seed(CONFIG["seed"] + epoch)
    return DataLoader(PolypDataset(SPLIT_IDS[split], training, epoch), batch_size=CONFIG["batch_size"],
                      shuffle=training, num_workers=CONFIG["workers"], generator=generator,
                      pin_memory=DEVICE.type == "cuda")

def mask_geometry(target, valid):
    target = target & valid
    # Four-connected exposed grid edges give a perimeter proxy in input pixels.
    padded = np.pad(target, 1).astype(np.int16)
    perimeter = np.abs(np.diff(padded, axis=0)).sum() + np.abs(np.diff(padded, axis=1)).sum()
    area = int(target.sum())
    components = int(ndimage.label(target)[1])
    return {
        "area_fraction": area / int(valid.sum()), "area_pixels": area,
        "boundary_complexity": float(perimeter ** 2 / (4 * np.pi * area)) if area else np.nan,
        "components": components,
        "touches_valid_edge": bool((target & ~ndimage.binary_erosion(valid)).any()),
    }

def build_model(pretrained=False):
    weights = LRASPP_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1 if pretrained else None
    try:
        model = lraspp_mobilenet_v3_large(weights=weights, weights_backbone=None)
    except (OSError, RuntimeError) as error:
        raise RuntimeError("TorchVision 사전학습 가중치/API 초기화 실패. 연결 상태와 "
                           "torch/torchvision 버전 조합을 확인하고 다시 실행하세요.") from error
    for name in ("low_classifier", "high_classifier"):
        old = getattr(model.classifier, name)
        setattr(model.classifier, name, nn.Conv2d(old.in_channels, 1, kernel_size=1))
    return model

def save_checkpoint(path, payload):
    temp = path.with_suffix(".pt.part")
    torch.save(payload, temp)
    temp.replace(path)

def metric_rows(predictions, targets, valid_pixels, keys, detailed=False):
    def ratio(numerator, denominator, empty):
        return numerator / denominator if denominator else float(empty)

    def boundary_f1(prediction, target, valid):
        a = prediction & ~ndimage.binary_erosion(prediction)
        b = target & ~ndimage.binary_erosion(target)
        if not a.any() or not b.any():
            return float(a.any() == b.any())
        tolerance = CONFIG["boundary_tolerance_px"]
        match_a = (a & valid & (ndimage.distance_transform_edt(~b) <= tolerance)).sum() / a.sum()
        match_b = (b & valid & (ndimage.distance_transform_edt(~a) <= tolerance)).sum() / b.sum()
        return ratio(2 * match_a * match_b, match_a + match_b, False)

    rows = []
    for key, pred, target, valid in zip(keys, predictions, targets, valid_pixels, strict=True):
        pred, target = pred & valid, target & valid
        tp = int((pred & target).sum())
        fp = int((pred & ~target).sum())
        fn = int((~pred & target).sum())
        tn = int(valid.sum() - tp - fp - fn)
        row = {
            "id": key, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "dice": ratio(2 * tp, 2 * tp + fp + fn, True),
            "iou": ratio(tp, tp + fp + fn, True),
            "recall": ratio(tp, tp + fn, True),
            "precision": ratio(tp, tp + fp, not target.any()),
        }
        if detailed:
            row.update(mask_geometry(target, valid))
            row["boundary_f1"] = boundary_f1(pred, target, valid)
            row["failure"] = row["dice"] < CONFIG["failure_dice"]
            row["missed_entire_polyp"] = bool(target.any() and not pred.any())
        rows.append(row)
    return rows

@torch.inference_mode()
def evaluate(model, split, detailed=False):
    model.eval()
    rows = []
    for images, targets, valid, keys in loader(split):
        probs = model(images.to(DEVICE, non_blocking=True))["out"].float().sigmoid().cpu()
        rows.extend(metric_rows(probs[:, 0].numpy() >= CONFIG["probability_threshold"],
                                targets[:, 0].numpy(), valid[:, 0].numpy(), keys, detailed))
    return pd.DataFrame(rows)

def objective(logits, target, valid, mode):
    target, valid = target.float(), valid.float()
    logits = logits.float()
    probability = logits.sigmoid() * valid
    dimensions = (1, 2, 3)
    area = target.sum(dim=dimensions)
    intersection = (probability * target).sum(dim=dimensions)
    dice_loss = 1.0 - ((2 * intersection + 1) / (probability.sum(dim=dimensions) + area + 1))
    weight = valid
    if mode == "small_positive":
        small = area / valid.sum(dim=dimensions) <= REFERENCE_CUTS["area_fraction"][0]
        emphasis = target * small[:, None, None, None]
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * emphasis)
    elif mode == "boundary":
        dilated = F.max_pool2d(target, 5, stride=1, padding=2)
        eroded = 1 - F.max_pool2d(F.pad(1 - target, (2, 2, 2, 2), value=1), 5, stride=1)
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * (dilated - eroded))
    pixel_bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (pixel_bce * weight).sum(dim=dimensions) / weight.sum(dim=dimensions)
    return (0.5 * bce + 0.5 * dice_loss).mean()

def png_mask(mask):
    with io.BytesIO() as buffer:
        Image.fromarray(mask.astype(np.uint8) * 255).save(buffer, format="PNG")
        return buffer.getvalue()

def smoke_inputs():
    # Build a small app-format fixture without declaring any simulated action human.
    folder = INPUTS / "generated_smoke"
    folder.mkdir(exist_ok=True)
    model_path = folder / "baseline_best.pt"
    seed_everything(CONFIG["seed"])
    model = build_model()
    # A compressible zero-initialized fixture exercises the pipeline. Actual review inputs retain their weights.
    with torch.no_grad():
        for parameter in model.parameters():
            parameter.zero_()
    record = {"mode": "baseline", "model": model.state_dict(), "run_id": "generated_smoke", "epoch": 0}
    save_checkpoint(model_path, record)
    setup = {"config": {**CONFIG, "profile": "smoke"}, "dataset": DATA_INFO, "split_sha256": SPLIT_SHA,
             "model": {"file": model_path.name, "model_run_id": "generated_smoke",
                       "sha256": sha256_file(model_path), "split_metadata_verified": True}}
    archive_path = folder / "review_results.zip"
    rows = []
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for index, key in enumerate(SPLIT_IDS["val_audit"][:3]):
            with Image.open(PAIR_BY_ID[key][1]) as image:
                width, height = image.size
            before = np.zeros((height, width), dtype=bool)
            before[height // 4:height // 2, width // 5:width // 2] = True
            after = before.copy()
            status = ("corrected", "approved", "deferred")[index]
            if status == "corrected":
                # An exaggerated edit makes the two optimization paths observable in a two-step smoke test.
                after[1:-1, 1:-1] = True
            before_path = f"masks/before/{key}.png"
            after_path = f"masks/after/{key}.png" if status != "deferred" else ""
            archive.writestr(before_path, png_mask(before))
            if after_path:
                archive.writestr(after_path, png_mask(after))
            rows.append({"case_id": key, "actor": "smoke_test", "status": status,
                         "before_mask": before_path, "after_mask": after_path,
                         "changed_pixels": int((before != after).sum()) if after_path else ""})
        archive.writestr("review_log.csv", pd.DataFrame(rows).to_csv(index=False))
        archive.writestr("review_setup.json", json.dumps(setup))
    return archive_path

if SMOKE and REVIEW_PATH is None:
    REVIEW_PATH = smoke_inputs()
    REVIEW_SETUP, REVIEW_LOG, REVIEW_SHA = read_review(REVIEW_PATH)
expected_checkpoint = REVIEW_SETUP["model"]["sha256"]
run_config = {"profile": PROFILE, "review_zip_sha256": REVIEW_SHA, "start_checkpoint_sha256": expected_checkpoint,
              "split_sha256": SPLIT_SHA, "config": CONFIG, "environment": ENV}
RUN_ID = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:12]
RUN = ROOT / "runs" / f"{PROFILE}_{RUN_ID}"
(RUN / "figures").mkdir(parents=True, exist_ok=True)

def get_mask(archive, member, original_size):
    if member not in archive.namelist() or archive.getinfo(member).file_size > 8 * 2**20:
        raise ValueError(f"필수 마스크가 없거나 크기가 너무 큽니다: {member}")
    payload = archive.read(member)
    try:
        with Image.open(io.BytesIO(payload)) as mask:
            if mask.format != "PNG" or mask.mode not in {"1", "L"} or mask.size != original_size:
                raise ValueError(f"{member}은 원본과 같은 크기 {original_size}의 단일 채널 PNG여야 합니다.")
            values = np.asarray(mask.convert("L"))
    except (OSError, Image.DecompressionBombError) as error:
        raise ValueError(f"마스크 파일을 열 수 없습니다: {member}") from error
    if not ((values == 0) | (values == 255)).all():
        raise ValueError(f"0 또는 255가 아닌 마스크 값이 있습니다: {member}")
    return values.astype(bool), payload

def validate_and_stage_reviews():
    actor = "smoke_test" if SMOKE else "human"
    selected = REVIEW_LOG.loc[REVIEW_LOG.actor.eq(actor)].copy()
    if not selected.status.isin(DECISIONS).all():
        raise ValueError("승인·수정·보류 외의 상태가 들어 있습니다.")
    if not selected.case_id.is_unique:
        raise ValueError("검수 로그에 중복 사례 ID가 있습니다.")
    selected = selected.sort_values("case_id")
    test_ids = set(SPLIT_IDS["test"])
    test_groups = {SPLIT_BY_ID[key]["group_id"] for key in test_ids}
    records, paired = [], {}
    if sha256_file(REVIEW_PATH) != REVIEW_SHA:
        raise ValueError("실행 중 검수 ZIP이 변경됐습니다. 내보내기를 완료한 뒤 다시 실행하세요.")
    with zipfile.ZipFile(REVIEW_PATH) as archive:
        for row in selected.itertuples(index=False):
            key = row.case_id
            if not CASE_RE.fullmatch(key) or key not in SPLIT_BY_ID:
                raise ValueError(f"Kvasir-SEG 분할에서 찾을 수 없는 사례 ID: {key}")
            split = SPLIT_BY_ID[key]
            if split["split"] != "val_audit" or key in test_ids or split["group_id"] in test_groups:
                raise ValueError(f"허용된 검수 분할이 아니거나 평가 사례와 겹칩니다: {key}")
            path = PAIR_BY_ID[key][1]
            with Image.open(path) as image:
                original_size = image.size
                image = image.convert("RGB")
            proposed = f"masks/before/{key}.png"
            final = f"masks/after/{key}.png" if row.status != "deferred" else ""
            if row.before_mask != proposed or row.after_mask != final:
                raise ValueError(f"{key}의 검수 전후 마스크 경로가 상태와 일치하지 않습니다.")
            before, before_bytes = get_mask(archive, proposed, original_size)
            changed_native, changed_model = None, None
            if final:
                after, after_bytes = get_mask(archive, final, original_size)
                changed_native = int((before != after).sum())
                if row.status == "approved" and changed_native:
                    raise ValueError(f"승인된 {key}의 마스크가 두 파일에서 다릅니다.")
                try:
                    change_matches = float(row.changed_pixels) == changed_native
                except ValueError:
                    change_matches = False
                if not change_matches:
                    raise ValueError(f"{key}의 changed_pixels 기록과 실제 PNG가 일치하지 않습니다.")
                # Calculate whether edits survive the exact training resize.
                _, a, valid = letterbox(image, Image.fromarray(before.astype(np.uint8) * 255), CONFIG["side"])
                _, b, _ = letterbox(image, Image.fromarray(after.astype(np.uint8) * 255), CONFIG["side"])
                changed_model = int(((a != b) & valid).sum())
                paths = []
                for filename, data in [(proposed, before_bytes), (final, after_bytes)]:
                    destination = RUN / "training_inputs" / filename
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    destination.write_bytes(data)
                    paths.append(destination)
                paired[key] = (path, *paths)
            records.append({"case_id": key, "group_id": split["group_id"], "split": split["split"],
                            "actor": row.actor, "status": row.status, "used_for_training": bool(final),
                            "changed_pixels_native": changed_native, "changed_pixels_model_grid": changed_model,
                            "width": original_size[0], "height": original_size[1]})
    cases = pd.DataFrame(records, columns=["case_id", "group_id", "split", "actor", "status", "used_for_training",
                                          "changed_pixels_native", "changed_pixels_model_grid", "width", "height"])
    return cases, paired

CASES, MATCHED = validate_and_stage_reviews()
save_csv(CASES, "reviewed_cases.csv")
COUNTS = Counter(CASES.status)
N_CHANGED = int(CASES.changed_pixels_native.fillna(0).gt(0).sum())
N_MODEL_CHANGED = int(CASES.changed_pixels_model_grid.fillna(0).gt(0).sum())
SKIP_REASON = None
if not N_CHANGED:
    SKIP_REASON = "검수 전후 PNG에서 실제로 픽셀이 달라진 사례가 0건입니다."
elif not N_MODEL_CHANGED:
    SKIP_REASON = "수정이 모델 입력으로 축소하는 과정에서 모두 사라져 두 학습 조건의 마스크가 같습니다."
print(f"승인 {COUNTS['approved']}장 · 수정 {COUNTS['corrected']}장 · 보류 {COUNTS['deferred']}장 · 실제 변경 {N_CHANGED}장")
if SKIP_REASON:
    print(f"재학습 생략 · {SKIP_REASON}")

## 05 · 같은 가중치에서 두 조건 학습

A는 처음 후보, B는 검수 후 후보를 사용합니다. 동일한 사진, 초기 가중치, 증강, 두 학습률과 업데이트 횟수를 사용합니다.

In [ ]:
def choose_checkpoint():
    if CHECKPOINT_PATH:
        candidates = [Path(CHECKPOINT_PATH).expanduser()]
    else:
        candidates = checkpoint_candidates()
    mismatches = []

    def match(paths):
        for path in paths:
            if not path.is_file():
                mismatches.append(f"{path.name} (파일 없음)")
                continue
            if path.stat().st_size > 80 * 2**20:
                mismatches.append(f"{path.name} (크기 초과)")
                continue
            digest = sha256_file(path)
            if digest == expected_checkpoint:
                return path
            mismatches.append(f"{path.name} ({digest[:12]}…)")
        return None

    checkpoint = match(candidates)
    if checkpoint is None and not CHECKPOINT_PATH and IN_COLAB:
        checkpoint = match(pick_upload(
            f"이 검수에 사용한 기준 모델을 선택하세요. 필요한 SHA-256은 {expected_checkpoint}입니다.", {".pt"}
        ))
    if checkpoint is None:
        details = ", ".join(mismatches[:8]) if mismatches else "후보 파일 없음"
        raise ValueError(
            f"review_setup.json에 기록된 모델 {REVIEW_SETUP['model'].get('file', 'baseline_best.pt')}이 필요합니다.\n"
            f"필요한 SHA-256: {expected_checkpoint}\n"
            f"확인한 후보: {details}\n"
            "검수 앱과 같은 런타임의 polyp_audit/runs 또는 kvasir_review/training 폴더에서 찾을 수 있습니다. "
            "검수 앱에 업로드했던 파일이면 그 원본 .pt를 사용하세요."
        )
    return checkpoint

def get_initial_state(path):
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != expected_checkpoint:
        raise ValueError("선택 후 가중치 파일이 변경됐습니다. 필요한 SHA-256: " + expected_checkpoint)
    checkpoint = torch.load(io.BytesIO(payload), weights_only=True, map_location="cpu")
    expected_run = REVIEW_SETUP["model"].get("model_run_id")
    if (not isinstance(checkpoint, dict) or checkpoint.get("mode") != "baseline" or "model" not in checkpoint
            or (expected_run and checkpoint.get("run_id") != expected_run)):
        raise ValueError("검수에 사용한 baseline_best.pt 형식 또는 실행 ID가 맞지 않습니다.")
    model = build_model()
    model.load_state_dict(checkpoint["model"], strict=True)
    return checkpoint["model"]

class ReviewedDataset(PolypDataset):
    def __init__(self, arm, epoch):
        # The parent class retains the original resize, normalization and seeded augmentation.
        slot = {"A": 1, "B": 2}[arm]
        self.pairs = [(key, MATCHED[key][0], MATCHED[key][slot]) for key in sorted(MATCHED)]
        self.augment, self.epoch = True, epoch

def fit_arm(arm, state):
    seed_everything(CONFIG["seed"])
    model = build_model()
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CONFIG["backbone_lr"]},
        {"params": model.classifier.parameters(), "lr": CONFIG["classifier_lr"]},
    ], weight_decay=CONFIG["weight_decay"])
    history, sequence = [], []
    same_inputs = hashlib.sha256()
    step = 0
    for epoch in range(1, CONFIG["epochs"] + 1):
        seed_everything(CONFIG["seed"] + epoch)
        model.train()
        # Tiny reviewed sets can make BN running statistics noisy.
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm2d):
                layer.eval()
        generator = torch.Generator().manual_seed(CONFIG["seed"] + epoch)
        batches = DataLoader(ReviewedDataset(arm, epoch), batch_size=CONFIG["batch_size"],
                             shuffle=True, generator=generator, num_workers=0)
        total_loss = 0.0
        for images, labels, valid, ids in batches:
            same_inputs.update(images.contiguous().numpy().tobytes())
            same_inputs.update(valid.contiguous().numpy().tobytes())
            sequence.append({"epoch": epoch, "step": step + 1, "ids": list(ids)})
            images, labels, valid = [value.to(DEVICE) for value in (images, labels, valid)]
            optimizer.zero_grad(set_to_none=True)
            loss = objective(model(images)["out"], labels, valid, "baseline")
            if not torch.isfinite(loss):
                raise FloatingPointError(f"{arm} 조건의 업데이트 {step + 1}에서 손실이 유한하지 않습니다.")
            loss.backward()
            optimizer.step()
            step += 1
            total_loss += float(loss.detach()) * len(ids)
        history.append({"arm": arm, "epoch": epoch, "mean_training_loss": total_loss / len(MATCHED),
                        "updates_so_far": step})
        print(f"{arm} · epoch {epoch}/{CONFIG['epochs']} · 업데이트 {step}")
    checkpoint_path = RUN / f"checkpoint_{arm}.pt"
    save_checkpoint(checkpoint_path, {"model": model.cpu().state_dict(), "arm": arm, "run_id": RUN_ID,
                                     "start_checkpoint_sha256": expected_checkpoint, "updates": step,
                                     "epoch": CONFIG["epochs"]})
    del model
    return {"path": checkpoint_path, "input_tensor_sha256": same_inputs.hexdigest(),
            "sequence": sequence, "step": step, "history": history}

FITS = None
if SKIP_REASON is None:
    BASELINE_PATH = choose_checkpoint()
    print(f"시작 가중치 확인 · {BASELINE_PATH.name} · SHA-256 일치")
    initial_state = get_initial_state(BASELINE_PATH)
    FITS = {arm: fit_arm(arm, initial_state) for arm in ("A", "B")}
    del initial_state
    for field in ("sequence", "step", "input_tensor_sha256"):
        if FITS["A"][field] != FITS["B"][field]:
            raise RuntimeError(f"A와 B의 통제 조건이 일치하지 않습니다: {field}")
    expected_steps = CONFIG["epochs"] * ((len(MATCHED) + CONFIG["batch_size"] - 1) // CONFIG["batch_size"])
    if FITS["A"]["step"] != expected_steps:
        raise RuntimeError("업데이트 횟수가 사전에 고정한 수와 다릅니다.")
    write_json(RUN / "matched_training_schedule.json",
               {field: FITS["A"][field] for field in ("sequence", "step", "input_tensor_sha256")})
    save_csv(pd.DataFrame(FITS["A"]["history"] + FITS["B"]["history"]), "training_history.csv")

## 06 · 마스크 품질과 모델 성능 평가

수정 마스크의 공개 참조 일치도와 재학습 모델의 기존 테스트셋 성능을 두 표로 기록합니다. 원래 테스트셋을 재사용하므로 탐색적 비교입니다.

In [ ]:
def paired_comparison(frame, metric, subgroup, seed):
    column = f"delta_{metric}"
    cluster = frame.groupby("group_id")[column].agg(["sum", "count"])
    n_groups = len(cluster)
    lower, upper = np.nan, np.nan
    if n_groups >= 2:
        rng = np.random.default_rng(seed)
        choices = rng.integers(n_groups, size=(CONFIG["bootstrap_repeats"], n_groups))
        bootstrap = cluster["sum"].to_numpy()[choices].sum(axis=1) / cluster["count"].to_numpy()[choices].sum(axis=1)
        lower, upper = np.quantile(bootstrap, [0.025, 0.975])
    return {
        "subgroup": subgroup, "metric": metric, "n": len(frame), "groups": n_groups,
        "baseline": frame[f"{metric}_baseline"].mean(), "intervention": frame[f"{metric}_intervention"].mean(),
        "delta": frame[column].mean(), "ci95_low": lower, "ci95_high": upper,
    }

METRICS = ("dice", "boundary_f1")
METRIC_NAMES = {"dice": "Dice", "boundary_f1": "Boundary F1"}

def pair_metrics(a, b):
    left, right = a[["id", *METRICS]], b[["id", *METRICS]]
    if not left.id.is_unique or not right.id.is_unique or set(left.id) != set(right.id):
        raise ValueError("두 조건에서 평가한 이미지가 일치하지 않습니다.")
    paired = left.merge(right, on="id", validate="one_to_one", suffixes=("_baseline", "_intervention"))
    paired["group_id"] = paired.id.map(lambda key: SPLIT_BY_ID[key]["group_id"])
    for metric in METRICS:
        paired[f"delta_{metric}"] = paired[f"{metric}_intervention"] - paired[f"{metric}_baseline"]
    return paired

def summary_of(paired, subgroup, labels):
    result = [paired_comparison(paired, metric, subgroup, CONFIG["seed"] + 850) for metric in METRICS]
    return pd.DataFrame(result).rename(columns={"baseline": labels[0], "intervention": labels[1]})

def public_mask_quality():
    scores = {"before": [], "after": []}
    for row in CASES.loc[CASES.changed_pixels_native.fillna(0).gt(0)].itertuples(index=False):
        key = row.case_id
        _, reference, valid = load_pair(*PAIR_BY_ID[key])
        image_path, before, after = MATCHED[key]
        for label, mask_path in {"before": before, "after": after}.items():
            _, proposal, _ = load_pair(key, image_path, mask_path)
            scores[label].extend(metric_rows([proposal], [reference], [valid], [key], detailed=True))
    a, b = (pd.DataFrame(scores[label]) for label in ("before", "after"))
    paired = pair_metrics(a, b)
    names = {f"{metric}_{key}": f"{metric}_{label}" for metric in METRICS
             for key, label in (("baseline", "before"), ("intervention", "after"))}
    save_csv(paired.rename(columns=names), "reviewed_mask_quality_per_image.csv")
    return summary_of(paired, "changed_review_cases", ("before", "after"))

def load_arm(arm):
    record = torch.load(FITS[arm]["path"], weights_only=True, map_location="cpu")
    if (record["arm"], record["run_id"], record["start_checkpoint_sha256"]) != (arm, RUN_ID, expected_checkpoint):
        raise ValueError(f"{arm} 체크포인트가 이 실행과 일치하지 않습니다.")
    model = build_model()
    model.load_state_dict(record["model"], strict=True)
    return model.to(DEVICE)

def evaluate_finetuned_models():
    observations = {}
    for arm in ("A", "B"):
        model = load_arm(arm)
        observations[arm] = evaluate(model, "test", detailed=True)
        del model
        if set(observations[arm].id) != set(SPLIT_IDS["test"]):
            raise ValueError("평가에 사용한 파일이 고정 테스트셋과 다릅니다.")
    paired = pair_metrics(observations["A"], observations["B"])
    names = {f"{metric}_{key}": f"{metric}_{label}" for metric in METRICS
             for key, label in (("baseline", "A"), ("intervention", "B"))}
    save_csv(paired.rename(columns=names), "test_paired_metrics.csv")
    return summary_of(paired, "original_test", ("A", "B"))

def plot_deltas(comparison, title, xlabel, filename):
    fig, ax = plt.subplots(figsize=(7.6, 3.3), layout="constrained")
    ax.axvline(0, color="#586574", linestyle="--", linewidth=1)
    for index, row in enumerate(comparison.itertuples(index=False)):
        if np.isfinite(row.ci95_low) and np.isfinite(row.ci95_high):
            ax.hlines(index, row.ci95_low, row.ci95_high, color="#107F79", lw=3)
        ax.plot(row.delta, index, "o", color="#107F79", markersize=7)
    ax.set(title=title, xlabel=xlabel, yticks=list(range(len(comparison))),
           yticklabels=[METRIC_NAMES[name] for name in comparison.metric], ylim=(-0.5, len(comparison)-0.5))
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
    save_figure(fig, filename)

QUALITY = None
if N_CHANGED:
    QUALITY = public_mask_quality()
    save_csv(QUALITY, "reviewed_mask_quality_summary.csv")
    plot_deltas(QUALITY, "Changed labels against the public reference",
                "After minus before (image-group bootstrap 95% interval)", "reviewed_mask_quality.png")

COMPARISON = None
if FITS is not None:
    COMPARISON = evaluate_finetuned_models()
    save_csv(COMPARISON, "test_comparison.csv")
    plot_deltas(COMPARISON, "Paired model comparison on the original test split",
                "B minus A (image-group bootstrap 95% interval)", "test_model_comparison.png")
    display(COMPARISON.round(4))

## 07 · 보고서와 결과 내보내기

ZIP의 `RESULTS.md`부터 읽으면 됩니다. 자동 다운로드가 시작되지 않으면 Colab 파일 탭에서 마지막에 표시된 경로를 찾으세요.

In [ ]:
def report_table(frame, first, second):
    lines = [f"| 지표 | {first} | {second} | 차이 | 짝지은 95% 구간 |",
             "| --- | ---: | ---: | ---: | --- |"]
    for row in frame.to_dict("records"):
        interval = (f"{row['ci95_low']:+.3f} ~ {row['ci95_high']:+.3f}"
                    if np.isfinite(row["ci95_low"]) and np.isfinite(row["ci95_high"])
                    else "계산 불가")
        lines.append(f"| {METRIC_NAMES[row['metric']]} | {row[first]:.3f} | {row[second]:.3f} | "
                     f"{row['delta']:+.3f} | {interval} |")
    return "\n".join(lines)

def inference_sentence(row):
    difference, low, high = row["delta"], row["ci95_low"], row["ci95_high"]
    label = METRIC_NAMES[row["metric"]]
    if not (np.isfinite(low) and np.isfinite(high)):
        return f"- **{label}** · 평가 그룹 수가 적어 불확실성 구간을 계산하지 못했습니다."
    if low <= 0 <= high:
        return f"- **{label}** · 95% 구간에 0이 포함되어 개선 효과가 불확실합니다."
    if abs(difference) < CONFIG["small_delta"]:
        return f"- **{label}** · 점수 차이의 절댓값이 0.01보다 작아 실질적 효과는 불확실합니다."
    direction = "상승" if difference > 0 else "하락"
    return f"- **{label}** · 이 비교에서 {difference:+.3f} {direction}했습니다."

def render_report():
    verified = REVIEW_SETUP["model"].get("split_metadata_verified") is True
    human = len(CASES) if not SMOKE else 0
    mode_name = "합성 데이터 CPU 스모크 테스트" if SMOKE else "검수 기록 기반 연구 파일럿"
    text = [
        "# 검수 전후 마스크로 이어서 학습한 용종 분할 모델 비교", "",
        f"**{mode_name}** · 실행 ID `{RUN_ID}` · {ENV['gpu'] or ENV['device']} · 입력 {CONFIG['side']}×{CONFIG['side']}.",
        "", "## 검수 입력", "",
        f"사람 검수 **{human}건**. {'아래 건수는 코드 점검용 합성 사례입니다.' if SMOKE else '아래 건수는 이번 검수 ZIP에서 선별한 실제 사용자 기록입니다.'}",
        f"승인 **{COUNTS['approved']}** · 수정 상태 **{COUNTS['corrected']}** · 보류 **{COUNTS['deferred']}** · 비교에 사용한 마스크 쌍 **{len(MATCHED)}**.",
        f"원본 PNG에서 실제 픽셀이 달라진 사례 **{N_CHANGED}** · 모델 입력 격자에서도 달라진 사례 **{N_MODEL_CHANGED}**.",
        f"다른 기록 유형으로 제외된 행 {len(REVIEW_LOG) - len(CASES)}건.",
        f"원래 기준 체크포인트의 학습 분할은 검수 앱에서 {'확인된 것으로 기록됐습니다.' if verified else '미확인으로 기록됐습니다. 원래 모델의 평가 분할 독립성은 미확인입니다.'}",
        "", "## 1. 수정 마스크와 공개 참조의 일치도", "",
    ]
    if QUALITY is None:
        text.append("실제로 수정된 마스크가 없어 이 비교의 대상이 0건입니다.")
    else:
        text.extend([
            f"실제 변경 사례 {N_CHANGED}장만 포함합니다. 공개 참조 마스크는 이 평가에서 사용했습니다.",
            "", report_table(QUALITY, "before", "after"), "",
            "![수정 전후 마스크 품질](figures/reviewed_mask_quality.png)",
        ])
    text.extend(["", "## 2. 이어서 학습한 모델의 성능", ""])
    if COMPARISON is None:
        text.append(f"**재학습 비교 생략.** {SKIP_REASON}")
    else:
        text.extend([
            f"동일한 기준 가중치에서 출발해 A에는 처음 마스크, B에는 승인·수정 후 마스크를 주었습니다. "
            f"두 조건 모두 같은 {len(MATCHED)}장, {CONFIG['epochs']} epoch, {FITS['A']['step']}회 업데이트로 학습했습니다.",
            "이미지 순서와 증강 결과의 해시도 두 조건에서 일치했습니다.",
            "", f"**{'스모크 테스트 평가' if SMOKE else '기존 테스트셋을 재사용한 탐색적 비교'}** · "
            f"동일한 평가 사진 {len(SPLIT_IDS['test'])}장.",
            "", report_table(COMPARISON, "A", "B"), "",
            *(["아래 표와 그래프의 수치는 합성 입력의 코드 동작을 점검하는 용도입니다."] if SMOKE else
              [inference_sentence(row) for row in COMPARISON.to_dict("records")]),
            "", "![두 학습 조건의 성능 차이](figures/test_model_comparison.png)",
        ])
    text.extend([
        "", "## 해석 범위", "",
        "Dice와 Boundary F1은 이미지별 점수의 평균입니다. Boundary F1의 허용 거리는 모델 입력 좌표계의 2픽셀입니다.",
        "95% 구간은 동일 이미지의 점수 차이를 연결한 뒤 이미지 그룹을 단위로 재표집해 계산했습니다. "
        "한 학습 시드에 조건부인 구간이며, 별도 재학습에서 생길 변동은 포함하지 않습니다.",
        "검수 사진과 평가 사진의 ID와 중복 그룹을 대조했고, 비교 설정은 평가 전에 고정했습니다. "
        "이미지 유사도 기반 분할에는 환자·검사 ID가 포함되지 않습니다.",
        "이 평가 범위는 공개 연구 데이터입니다. 사용자 검수 마스크의 전문가 간 일치도는 별도 측정이 필요합니다.",
        "", f"Jha et al. (2020). Kvasir-SEG. [공식 배포]({OFFICIAL_PAGE}) · [논문]({PAPER})", "",
    ])
    return "\n".join(text)

run_setup = {
    **run_config, "run_id": RUN_ID, "dataset": DATA_INFO,
    "input_review_filename": REVIEW_PATH.name, "review_model": REVIEW_SETUP["model"],
    "checkpoint_sha256_verified": FITS is not None,
    "review_decisions": {status: COUNTS[status] for status in DECISIONS}, "changed_native_cases": N_CHANGED,
    "changed_model_grid_cases": N_MODEL_CHANGED, "skip_reason": SKIP_REASON,
    "training_ids": sorted(MATCHED), "evaluation_ids": SPLIT_IDS["test"] if FITS else [],
    "fixed_comparison": {"loss": "0.5 BCE + 0.5 Dice", "precision": "FP32",
                         "batchnorm_running_stats": "frozen", "selection": "fixed last epoch",
                         "training_data": "val_audit approved/corrected pairs, matched by ID",
                         "test_data": "original test; reused exploratory comparison",
                         "bootstrap_unit": "image-similarity duplicate group"},
}
write_json(RUN / "run_setup.json", run_setup)
(RUN / "RESULTS.md").write_text(render_report(), encoding="utf-8")

includes = ["RESULTS.md", "run_setup.json", "reviewed_cases.csv"]
if QUALITY is not None:
    includes.extend(["reviewed_mask_quality_summary.csv", "reviewed_mask_quality_per_image.csv",
                     "figures/reviewed_mask_quality.png"])
if COMPARISON is not None:
    includes.extend(["checkpoint_A.pt", "checkpoint_B.pt", "training_history.csv", "matched_training_schedule.json",
                     "test_comparison.csv", "test_paired_metrics.csv", "figures/test_model_comparison.png"])
RESULT_ARCHIVE = RUN / f"kvasir_finetune_results_{PROFILE}_{RUN_ID}.zip"
with tempfile.NamedTemporaryFile(dir=RUN, suffix=".zip.part", delete=False) as temporary:
    name = Path(temporary.name)
try:
    with zipfile.ZipFile(name, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
        for member in includes:
            archive.write(RUN / member, member)
    with zipfile.ZipFile(name) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("결과 ZIP 무결성 검사에 실패했습니다.")
    name.replace(RESULT_ARCHIVE)
finally:
    name.unlink(missing_ok=True)

display(Markdown((RUN / "RESULTS.md").read_text(encoding="utf-8").split("## 해석 범위", 1)[0]
                 .replace("![수정 전후 마스크 품질](figures/reviewed_mask_quality.png)", "")
                 .replace("![두 학습 조건의 성능 차이](figures/test_model_comparison.png)", "")))
print(f"보고서 · {RUN / 'RESULTS.md'}")
print(f"결과 ZIP · {RESULT_ARCHIVE}")
if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    try:
        files.download(str(RESULT_ARCHIVE))
    except Exception as error:
        print(f"자동 다운로드가 시작되지 않았습니다. Colab 파일 탭에서 ZIP을 내려받으세요. {error}")